# 01 — Source audit and manifest

Inspect the declared sources, freeze the source registry, and make source roles explicit before downloading anything.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

import yaml
from portugal_refining_resilience.sources import load_source_manifest


In [ ]:
sources = load_source_manifest(ROOT / "config" / "sources.yml")
source_df = pd.DataFrame([{"source_id": key, **value} for key, value in sources.items()])
display(source_df[[c for c in ["source_id", "organisation", "role", "coverage"] if c in source_df.columns]])
persist_dataframe(source_df, PATHS.provenance / "source_manifest_snapshot.csv", key_columns=["source_id"])


## Audit rule

Primary analytical quantities should be independently cross-checked wherever feasible. DGEG is the preferred Portugal-specific reference; JODI provides transparent monthly product/flow aggregation; Eurostat supplies harmonised annual balances and the Spain comparison; the EC Weekly Oil Bulletin supplies comparable price histories.


In [ ]:
# The price results moved materially between drafts on a sample correction, and an
# estimator's defaults can move them the same way. Checksums on outputs establish that
# a file has not changed; they say nothing about what produced it.
import platform
import sys as _sys

import numpy as _np
import pandas as _pd
import scipy as _scipy
import statsmodels as _statsmodels

environment = pd.DataFrame(
    [
        {"component": "python", "version": platform.python_version()},
        {"component": "numpy", "version": _np.__version__},
        {"component": "pandas", "version": _pd.__version__},
        {"component": "scipy", "version": _scipy.__version__},
        {"component": "statsmodels", "version": _statsmodels.__version__},
        {"component": "platform", "version": platform.platform(terse=True)},
        {"component": "implementation", "version": platform.python_implementation()},
    ]
)
persist_dataframe(environment, PATHS.provenance / "software_environment.csv", key_columns=["component"])
display(environment)
